In [8]:
!pip install transformers underthesea torch numpy pandas json sklearn

ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json


In [9]:
import json
import numpy as np
import torch
import os
import csv
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from underthesea import ner, word_tokenize
import warnings

In [10]:
# Tắt cảnh báo khi tính PPL
warnings.filterwarnings("ignore", category=FutureWarning)

# =================================================================
#               KHỞI TẠO MÔ HÌNH VÀ TOKENIZER
# =================================================================

# Sử dụng PhoBERT-base làm mô hình nhúng (EMBEDDING)
EMB_MODEL_NAME = "vinai/phobert-base"
tokenizer_emb = AutoTokenizer.from_pretrained(EMB_MODEL_NAME)
model_emb = AutoModel.from_pretrained(EMB_MODEL_NAME)

# Mô hình cho Perplexity (PPL)
# SỬ DỤNG PHOBERT-BASE CŨNG CHO PPL (Cần lưu ý tính ổn định khi dùng MLM cho PPL)
# Để đơn giản hóa, tôi sẽ sử dụng mô hình EMB và tokenizer EMB cho PPL, 
# nhưng lý tưởng là bạn nên dùng một mô hình Causal LM.
tokenizer_ppl = tokenizer_emb
model_ppl = model_emb # Sử dụng model_emb cho cả PPL để tránh lỗi tải mô hình

# Đặt thiết bị tính toán (GPU nếu có)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_emb.to(DEVICE)
model_ppl.to(DEVICE)

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(64001, 768, padding_idx=1)
    (position_embeddings): Embedding(258, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropou

In [11]:

# def get_embedding(tokenizer, model, text, qid=None, subject=None, max_len=512):
#     if not text.strip():
#         return np.zeros(768)
#     try:
#         inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=max_len)
#         with torch.no_grad():
#             outputs = model(**inputs)
#         return outputs.last_hidden_state[:, 0, :].squeeze().numpy()  # CLS
#     except Exception as e:
#         print(f"Lỗi khi embedding câu hỏi id={qid}, môn={subject}")
#         return np.zeros(768)

def get_embedding(tokenizer, model, text, qid=None, subject=None, max_len=512):
    """Tính toán Semantic Embedding (CLS Token) bằng mô hình Transformer."""
    if not text.strip():
        return np.zeros(768)
    try:
        clean_text = text.replace("_", " ")
        inputs = tokenizer(clean_text, return_tensors="pt", truncation=True, padding=True, max_length=max_len).to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs)
        return outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    except Exception as e:
        print(f"Lỗi khi embedding câu hỏi id={qid}, môn={subject}: {e}")
        return np.zeros(768)

In [12]:
def calculate_jaccard_sim(text1, text2):
    """Tính Jaccard Similarity (Tương đồng Từ vựng)."""
    tokens1 = set(text1.replace("_", " ").split())
    tokens2 = set(text2.replace("_", " ").split())
    if not tokens1 and not tokens2:
        return 1.0
    if not tokens1 or not tokens2:
        return 0.0
    return len(tokens1.intersection(tokens2)) / len(tokens1.union(tokens2))

Đoạn code dưới đây sẽ trích xuất ra độ nhiễu của giữa từng options so với answer:   
- **mean_sim**: trung bình cosine similarity giữa đáp án đúng và các đáp án sai. Càng cao càng giống
- **max_sim**: giá trị similarity lớn nhất giữa đáp án đúng và các đáp án sai  
- **min_sim**: giá trị similarity nhỏ nhất  
- **std_sim**: độ lệch chuẩn similarity. Nếu std_sim cao → có option gần đúng và option rất xa đúng cùng tồn tại, câu hỏi có thể gây phân vân; nếu thấp → tất cả option sai đều gần như cùng mức so với đáp án đúng.
- **range_sim**: khoảng cách giữa max_sim và min_sim. một số option rất gần đáp án đúng, số khác rất xa, gây phân tán độ khó.


In [13]:
def calculate_entity_overlap(text1, text2):
    """Tính tỷ lệ trùng lặp thực thể (Entity Overlap)."""
    text1_clean = text1.replace("_", " ")
    text2_clean = text2.replace("_", " ")
    
    # Chỉ xem xét các thực thể chính
    entities1 = set([item[0] for item in ner(text1_clean) if item[1] in ['PERSON', 'LOCATION', 'ORGANIZATION', 'EVENT']])
    entities2 = set([item[0] for item in ner(text2_clean) if item[1] in ['PERSON', 'LOCATION', 'ORGANIZATION', 'EVENT']])
    
    if not entities1 and not entities2:
        return 0.0
    if not entities1 or not entities2:
        return 0.0
    
    return len(entities1.intersection(entities2)) / len(entities1.union(entities2))

In [14]:
def calculate_ppl(tokenizer, model, text, max_len=512):
    """Tính Perplexity (PPL). Sẽ dùng Cross-Entropy Loss."""
    if not text.strip():
        return 10000.0
    
    try:
        clean_text = text.replace("_", " ")
        encodings = tokenizer(clean_text, return_tensors='pt', truncation=True, max_length=max_len)
        
        # Tạo labels từ input_ids
        input_ids = encodings.input_ids.to(DEVICE)
        labels = input_ids.clone()
        
        # Tính toán loss (cross-entropy)
        with torch.no_grad():
            outputs = model(input_ids, labels=labels)
            loss = outputs.loss
            
        return torch.exp(loss).item()
    except Exception as e:
        # Lỗi có thể xảy ra nếu mô hình không phải Causal LM
        # print(f"Lỗi tính PPL: {e}. Dùng giá trị an toàn.") 
        text_len = len(text.split())
        return max(100.0, text_len * 10.0) # Giá trị an toàn

# Các từ khóa Cấu trúc (Structure)
WH_WORDS = {
    "is_Why": ["Tại sao", "Lý do", "Nguyên nhân"],
    "is_How": ["Như thế nào", "Cách thức", "Biện pháp"],
    "is_When": ["Khi nào", "Thời điểm", "Năm"],
    "is_Compare": ["So sánh", "Điểm khác biệt", "Giống nhau"]
}
NEGATION_WORDS = ["không", "ngoại trừ", "sai", "chưa", "ít", "ko", "chứ không phải"]

In [15]:
def extract_structure_features(question_stem):
    """Trích xuất các features cấu trúc câu hỏi (Wh-words, Phủ định)."""
    features = {}
    q_lower = question_stem.lower()
    
    for key, words in WH_WORDS.items():
        features[key] = 1 if any(word.lower() in q_lower for word in words) else 0
        
    features["is_Negation"] = 1 if any(word.lower() in q_lower for word in NEGATION_WORDS) else 0
    
    return features

In [16]:


# def get_diff(input, output, tokenizer, model):
#     features = []
#     ids = []
#     csv_rows = []  

#     with open(input, "r", encoding="utf-8") as f:
#         data = json.load(f)
    
#     for item in tqdm(data):
#         question = item["question"]
#         options = item["options"]
#         answer_key = item["answer"]

#         try:
#             answer_index = options.index(answer_key)
#         except ValueError:
#             print(f"Câu hỏi {item['id']} không có answer trong options")
#             continue 
        
#         emb_answer = get_embedding(tokenizer, model, question + " " + options[answer_index], qid=item["id"], subject=item["subject"])
#         if emb_answer is None or not emb_answer.any():
#             continue

#         wrong_embs = [
#             get_embedding(tokenizer, model, question + " " + opt, qid=item["id"], subject=item["subject"])
#             for i, opt in enumerate(options) if i != answer_index
#         ]
#         sims = [cosine_similarity([emb_answer], [emb])[0][0] for emb in wrong_embs]

#         if len(sims) > 0:
#             mean_sim = float(np.mean(sims))
#             max_sim = float(np.max(sims))
#             min_sim = float(np.min(sims))
#             std_sim = float(np.std(sims))
#             range_sim = max_sim - min_sim
#         else:
#             mean_sim = max_sim = min_sim = std_sim = range_sim = 0.0

#         features.append([mean_sim, max_sim, min_sim, std_sim, range_sim])
#         ids.append(item["id"])
#         csv_rows.append({
#             "id": item["id"],
#             "question": item["question"],
#             "answer": answer_key,
#             "mean_sim": mean_sim,
#             "max_sim": max_sim,
#             "min_sim": min_sim,
#             "std_sim": std_sim,
#             "range_sim": range_sim
#         })
#     features = np.array(features) 
#     ids = np.array(ids)

#     os.makedirs(output, exist_ok=True)
#     csv_path = os.path.join(output, "noise_features.csv")
#     with open(csv_path, "w", newline='', encoding='utf-8') as f:
#         writer = csv.DictWriter(f, fieldnames=csv_rows[0].keys())
#         writer.writeheader()
#         writer.writerows(csv_rows)

#     print("--------Đã hoàn thành xong feartures tính độ lệch nhau giữa các options-------------\n", csv_path)

# Hàm get_diff sẽ sử dụng các mô hình/tokenizer đã khởi tạo ở phạm vi ngoài
def get_diff(input_file, output_dir):
    
    # ------------------ KHỞI TẠO TÊN CỘT FEATURES ------------------
    base_cols = ["id", "subject", "question", "answer"]
    sim_cols = ["mean_semantic_sim", "max_sim", "min_sim", "std_sim", "range_sim",
                "mean_jaccard_sim", "std_jaccard_sim",
                "mean_entity_overlap", "max_entity_overlap",
                "mean_inter_semantic_sim", "std_inter_semantic_sim"]
    ppl_cols = ["PPL_Stem", "PPL_Option_Mean", "PPL_Gap", "PPL_Range"]
    structure_cols = list(WH_WORDS.keys()) + ["is_Negation"]
    
    fieldnames = base_cols + sim_cols + ppl_cols + structure_cols
    # ----------------------------------------------------------------

    all_features = []
    
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    print(f"---------Đang tính toán độ nhiễu nâng cao cho {len(data)} câu hỏi-------------\n")
    
    for item in tqdm(data):
        question = item["question"]
        options = item["options"]
        answer_key = item["answer"]
        qid = item["id"]
        subject = item.get("subject", "Unknown")
        
        current_row = {"id": qid, "subject": subject, "question": question, "answer": answer_key}

        # --- I. Trích xuất Feature Cấu trúc & PPL Nâng cao ---
        structure_feats = extract_structure_features(question)
        current_row.update(structure_feats)
        
        # Tính PPL
        ppl_stem = calculate_ppl(tokenizer_ppl, model_ppl, question)
        ppl_options = [calculate_ppl(tokenizer_ppl, model_ppl, opt) for opt in options]
        
        current_row["PPL_Stem"] = ppl_stem
        if ppl_options:
            ppl_option_mean = np.mean(ppl_options)
            current_row["PPL_Option_Mean"] = ppl_option_mean
            current_row["PPL_Range"] = np.max(ppl_options) - np.min(ppl_options)
            current_row["PPL_Gap"] = abs(ppl_stem - ppl_option_mean)
        else:
            current_row["PPL_Option_Mean"] = current_row["PPL_Stem"]
            current_row["PPL_Range"] = current_row["PPL_Gap"] = 0.0
            
        # --- II. Trích xuất Feature Tương đồng Đa tầng ---

        try:
            answer_index = options.index(answer_key)
            correct_option_text = options[answer_index]
        except ValueError:
            # print(f"Câu hỏi {qid} không có answer trong options")
            continue 
        
        # 1. Chuẩn bị Sim
        jaccard_sims, entity_overlaps, semantic_sims = [], [], []
        wrong_embs = []
        
        emb_answer = get_embedding(tokenizer_emb, model_emb, question + " " + correct_option_text, qid=qid, subject=subject)
        if not emb_answer.any():
            continue

        for i, opt in enumerate(options):
            if i != answer_index:
                
                # Tầng 3: Semantic Sim
                emb_distractor = get_embedding(tokenizer_emb, model_emb, question + " " + opt, qid=qid, subject=subject)
                wrong_embs.append(emb_distractor)
                
                sim = cosine_similarity([emb_answer.reshape(1, -1)], [emb_distractor.reshape(1, -1)])[0][0]
                semantic_sims.append(sim)
                
                # Tầng 1: Lexical (Jaccard)
                j_sim = calculate_jaccard_sim(correct_option_text, opt)
                jaccard_sims.append(j_sim)
                
                # Tầng 2: Entity
                e_overlap = calculate_entity_overlap(correct_option_text, opt)
                entity_overlaps.append(e_overlap)

        # 2. Tổng hợp Sim giữa Đúng và Sai
        if semantic_sims:
            # Semantic (Features ban đầu)
            current_row["mean_semantic_sim"] = float(np.mean(semantic_sims))
            current_row["max_sim"] = float(np.max(semantic_sims))
            current_row["min_sim"] = float(np.min(semantic_sims))
            current_row["std_sim"] = float(np.std(semantic_sims))
            current_row["range_sim"] = current_row["max_sim"] - current_row["min_sim"]
            
            # Lexical (Mới)
            current_row["mean_jaccard_sim"] = float(np.mean(jaccard_sims))
            current_row["std_jaccard_sim"] = float(np.std(jaccard_sims))
            
            # Entity (Mới)
            current_row["mean_entity_overlap"] = float(np.mean(entity_overlaps))
            current_row["max_entity_overlap"] = float(np.max(entity_overlaps))
            
        else: # Điền 0.0 cho các features Sim
            pass

        # 3. Tương đồng giữa các Nhiễu (Inter-Distractor Semantic Sim)
        inter_sims = []
        num_wrong = len(wrong_embs)
        for i in range(num_wrong):
            for j in range(i + 1, num_wrong):
                sim = cosine_similarity([wrong_embs[i].reshape(1, -1)], [wrong_embs[j].reshape(1, -1)])[0][0]
                inter_sims.append(sim)
        
        if inter_sims:
            current_row["mean_inter_semantic_sim"] = float(np.mean(inter_sims))
            current_row["std_inter_semantic_sim"] = float(np.std(inter_sims))
        else:
            current_row["mean_inter_semantic_sim"] = 0.0
            current_row["std_inter_semantic_sim"] = 0.0
            
        all_features.append(current_row)

    # --- III. Lưu kết quả ---
    os.makedirs(output_dir, exist_ok=True)
    csv_path = os.path.join(output_dir, "noise_features_advanced.csv")
    
    # Đảm bảo tất cả rows đều có đủ fieldnames (điền 0.0 cho các cột Sim nếu thiếu)
    processed_rows = []
    for row in all_features:
        # Sử dụng get() với giá trị mặc định 0.0 cho các cột Sim/PPL/Structure nếu chúng không được tính (do câu hỏi lỗi)
        full_row = {col: row.get(col, 0.0) for col in fieldnames}
        # Đảm bảo các cột string vẫn là string
        full_row["id"] = row.get("id")
        full_row["subject"] = row.get("subject")
        full_row["question"] = row.get("question")
        full_row["answer"] = row.get("answer")
        processed_rows.append(full_row)
        
    with open(csv_path, "w", newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(processed_rows)

    print(f"--------Đã hoàn thành xong feartures tính độ nhiễu nâng cao-------------\n", csv_path)
    print(f"Tổng số câu hỏi được xử lý: {len(processed_rows)}")
    
    return csv_path

---------Đang tính toán độ lệch embedding giữa các options-------------

---------Đang tính toán độ nhiễu nâng cao cho 3097 câu hỏi-------------



  0%|          | 0/3097 [00:00<?, ?it/s]


ValueError: Found array with dim 3, while dim <= 2 is required by check_pairwise_arrays.